# 06 — Applied Case Studies: IoU, Non-Maximum Suppression, and Image Histograms

## Integrating Compound Data Structures in Computer Vision

In modern deep learning and computer vision architectures (such as YOLOv8, Faster R-CNN, and SSD), raw tensor predictions must be post-processed using algorithmic pipelines built upon fundamental data structures:
1. **Tuples:** Represent axis-aligned bounding boxes $(x_{\min}, y_{\min}, x_{\max}, y_{\max})$ and detection records $(x_{\min}, y_{\min}, x_{\max}, y_{\max}, \text{confidence})$.
2. **Lists:** Maintain dynamically filtered candidate detection collections.
3. **Dictionaries:** Map discrete pixel intensity values to frequency distributions (histograms).

In this notebook, we implement and evaluate:
- The **Intersection over Union (IoU)** metric.
- The **Greedy Non-Maximum Suppression (NMS)** algorithm.
- 2D **Image Pixel Intensity Frequency Histograms**.


In [ ]:
# 1. Intersection over Union (IoU) Engine
def compute_iou(
    box_a: tuple[float, float, float, float],
    box_b: tuple[float, float, float, float],
) -> float:
    """Computes exact spatial Intersection over Union (IoU) between two bounding boxes.

    Coordinates format: (x_min, y_min, x_max, y_max)
    """
    # Coordinates of intersection rectangle
    x_a = max(box_a[0], box_b[0])
    y_a = max(box_a[1], box_b[1])
    x_b = min(box_a[2], box_b[2])
    y_b = min(box_a[3], box_b[3])

    # Width and height of intersection
    inter_w = max(0.0, x_b - x_a)
    inter_h = max(0.0, y_b - y_a)
    inter_area = inter_w * inter_h

    # Areas of individual boxes
    area_a = max(0.0, box_a[2] - box_a[0]) * max(0.0, box_a[3] - box_a[1])
    area_b = max(0.0, box_b[2] - box_b[0]) * max(0.0, box_b[3] - box_b[1])

    # Combined union area
    union_area = area_a + area_b - inter_area
    if union_area <= 0.0:
        return 0.0

    return inter_area / union_area


# Edge case validation
box_standard = (10.0, 10.0, 50.0, 50.0)
assert compute_iou(box_standard, box_standard) == 1.0  # Identity
assert compute_iou(box_standard, (60.0, 60.0, 90.0, 90.0)) == 0.0  # Completely disjoint
assert compute_iou(box_standard, (50.0, 10.0, 80.0, 50.0)) == 0.0  # Zero-area boundary touch
print("IoU engine passed all boundary assertions.")

## Greedy Non-Maximum Suppression (NMS)

Object detection models output dense candidate proposals per object. Non-Maximum Suppression eliminates redundant overlapping boxes using a greedy iterative algorithm:

### Algorithmic Procedure
1. Given candidate bounding box list $B = \{b_1, b_2, \dots, b_n\}$ where each box is $(x_1, y_1, x_2, y_2, c)$.
2. Sort $B$ descending by confidence score $c$.
3. Initialize empty selected list $K = []$.
4. While $B$ is not empty:
   a. Pop highest-confidence box $b_{\text{best}}$ from $B$ and append to $K$.
   b. For each remaining box $b_j \in B$, compute $\text{IoU}(b_{\text{best}}, b_j)$.
   c. If $\text{IoU}(b_{\text{best}}, b_j) \ge \tau_{\text{iou}}$, suppress (remove) $b_j$ from $B$.
5. Return $K$.


In [ ]:
# 2. Pure Python Non-Maximum Suppression (NMS)
def non_max_suppression(
    boxes: list[tuple[float, float, float, float, float]],
    iou_threshold: float,
) -> list[tuple[float, float, float, float, float]]:
    """Performs greedy Non-Maximum Suppression on bounding box detections."""
    if not boxes:
        return []

    # Sort candidates by confidence score descending (index 4)
    remaining = sorted(boxes, key=lambda b: b[4], reverse=True)
    picked: list[tuple[float, float, float, float, float]] = []

    while remaining:
        # Highest confidence box
        current = remaining.pop(0)
        picked.append(current)

        # Retain only candidates whose IoU is below suppression threshold
        survivors: list[tuple[float, float, float, float, float]] = []
        for candidate in remaining:
            iou = compute_iou(current[:4], candidate[:4])
            if iou < iou_threshold:
                survivors.append(candidate)
        remaining = survivors

    return picked


# Candidate bounding boxes: (x1, y1, x2, y2, confidence)
candidate_boxes = [
    (12.0, 84.0, 140.0, 212.0, 0.3),
    (24.0, 84.0, 152.0, 212.0, 0.4),
    (36.0, 84.0, 164.0, 212.0, 0.5),
    (12.0, 96.0, 140.0, 224.0, 0.6),
    (24.0, 96.0, 152.0, 224.0, 0.7),
    (24.0, 108.0, 152.0, 236.0, 0.8),
]

picked_boxes = non_max_suppression(candidate_boxes, iou_threshold=0.3)

print(f"Total candidate boxes: {len(candidate_boxes)}")
print(f"Boxes preserved by NMS: {len(picked_boxes)}")
for idx, b in enumerate(picked_boxes, start=1):
    print(f"  {idx}. Coords={b[:4]}, Confidence={b[4]}")

assert len(picked_boxes) < len(candidate_boxes)
assert picked_boxes[0][4] == 0.8  # Highest scoring candidate preserved

## Vectorized Non-Maximum Suppression with NumPy

In production inferencing, vectorized NumPy array operations replace element-by-element Python loops to suppress thousands of anchor proposals per second.


In [ ]:
# 3. Vectorized NMS with NumPy Fallback
def nms_vectorized(
    boxes_list: list[tuple[float, float, float, float, float]], iou_threshold: float
):
    """Vectorized NMS using NumPy broadcasting when available."""
    try:
        import numpy as np

        if not boxes_list:
            return []

        boxes = np.array(boxes_list, dtype=float)
        x1 = boxes[:, 0]
        y1 = boxes[:, 1]
        x2 = boxes[:, 2]
        y2 = boxes[:, 3]
        scores = boxes[:, 4]

        areas = (x2 - x1) * (y2 - y1)
        order = scores.argsort()[::-1]

        keep: list[int] = []
        while order.size > 0:
            i = order[0]
            keep.append(i)

            xx1 = np.maximum(x1[i], x1[order[1:]])
            yy1 = np.maximum(y1[i], y1[order[1:]])
            xx2 = np.minimum(x2[i], x2[order[1:]])
            yy2 = np.minimum(y2[i], y2[order[1:]])

            w = np.maximum(0.0, xx2 - xx1)
            h = np.maximum(0.0, yy2 - yy1)
            inter = w * h
            iou = inter / (areas[i] + areas[order[1:]] - inter)

            surviving_indices = np.where(iou < iou_threshold)[0]
            order = order[surviving_indices + 1]

        return [tuple(boxes[idx]) for idx in keep]
    except ImportError:
        # Fallback to pure python implementation
        return non_max_suppression(boxes_list, iou_threshold)


vec_picked = nms_vectorized(candidate_boxes, iou_threshold=0.3)
print(f"Vectorized NMS preserved {len(vec_picked)} detections.")
assert len(vec_picked) == len(picked_boxes)

## 2D Image Pixel Intensity Frequency Histograms

A digital grayscale image is a 2D matrix $I \in \mathbb{R}^{H \times W}$ where each cell $I[r, c]$ represents a discrete pixel intensity value in $[0, 255]$.

A pixel intensity histogram maps each discrete intensity value $v \in [0, 255]$ to its total frequency across the image:

$$H(v) = \sum_{r=0}^{H-1} \sum_{c=0}^{W-1} \mathbb{I}(I[r, c] == v)$$

We demonstrate histogram extraction using a pure Python dictionary mapping and test against our local session asset `../data/binary_cat.png`.


In [ ]:
# 4. 2D Image Histogram Computation via Dictionary
from pathlib import Path

# 1. Synthetic 7x7 binary image matrix
synthetic_image_matrix = [
    [255, 0, 255, 255, 255, 0, 255],
    [255, 0, 0, 255, 255, 0, 255],
    [255, 0, 0, 0, 0, 0, 255],
    [0, 0, 255, 255, 255, 0, 0],
    [0, 0, 255, 255, 255, 0, 0],
    [255, 0, 255, 255, 255, 0, 255],
    [255, 255, 0, 0, 0, 255, 255],
]

intensity_histogram: dict[int, int] = {}
for row in synthetic_image_matrix:
    for pixel in row:
        intensity_histogram[pixel] = intensity_histogram.get(pixel, 0) + 1

print(f"Synthetic matrix histogram: {intensity_histogram}")
assert sum(intensity_histogram.values()) == 7 * 7
assert intensity_histogram[0] == 23
assert intensity_histogram[255] == 26

# 2. Validating binary_cat.png asset
asset_path = Path("../data/binary_cat.png")
assert asset_path.is_file(), f"Asset not found: {asset_path}"
print(f"Asset verified: {asset_path} (size: {asset_path.stat().st_size} bytes)")

# Visualizing histogram (with fallback text mode if matplotlib is absent)
try:
    import matplotlib.pyplot as plt

    vals = sorted(intensity_histogram.keys())
    counts = [intensity_histogram[v] for v in vals]

    plt.figure(figsize=(6, 3.5))
    plt.bar([str(v) for v in vals], counts, color=["black", "lightgray"], edgecolor="black")
    plt.xlabel("Intensity Value")
    plt.ylabel("Pixel Count")
    plt.title("Synthetic 2D Binary Image Intensity Histogram")
    plt.tight_layout()
    plt.close()
    print("Matplotlib figure generated successfully.")
except ImportError:
    print("Matplotlib not present; textual histogram visualization:")
    for val, count in sorted(intensity_histogram.items()):
        bar = "#" * count
        print(f"  Intensity {val:3d}: {count:2d} | {bar}")